# MetaPulsar Usage Example

This notebook demonstrates how to use MetaPulsar to combine pulsar timing data from multiple PTA collaborations into unified Enterprise pulsar objects.

## Overview

The workflow covers:
1. **Manual single-pulsar data preparation** - Creating data structures by hand
2. **MetaPulsar creation** - Using consistent parameter merging strategies
3. **Enterprise integration** - Working with the resulting pulsar objects
4. **Automated discovery** - Processing multiple pulsars automatically
5. **Reference PTA selection** - Different strategies for choosing reference PTAs

## Key Concepts

- **Reference PTA**: The PTA whose parameters are inherited by the MetaPulsar for merged model components
- **Consistent Strategy**: Merges compatible parameters from different PTAs where possible
- **Component Merging**: Controls which parameter types (astrometry, spindown, binary, dispersion) are merged
- **Parameter Naming**: Merged parameters have no suffix, PTA-specific parameters retain PTA suffixes


In [ ]:
import loguru
import sys
from metapulsar import (
    create_metapulsar,
    discover_files,
    discover_layout,
    combine_layouts,
)

# Suppress debug output for cleaner example
loguru.logger.remove()
loguru.logger.add(sys.stdout, level="WARNING")


## Part 1: Manual Single-Pulsar Data Preparation

For single pulsars, manually creating the data structure is often the most transparent and flexible approach. This gives you full control over which PTAs to include and their ordering. It is a bit of work though, so we will go over more convenient methods later

### Data Structure

The data structure is a dictionary where:
- **Keys**: PTA names (e.g., 'nanograv_9y', 'epta_dr2')
- **Values**: Lists of file information dictionaries
- **Reference PTA**: The first PTA in the dictionary becomes the reference PTA

### File Information Fields
- `par`: Path to .par file (pulsar parameters)
- `tim`: Path to .tim file (timing observations)  
- `timing_package`: Software used (tempo2/pint)


In [ ]:
# Manually create a single-pulsar dictionary with three PTAs
# The reference PTA (first in the dictionary) will be used for parameter
# inheritance where appropriate
pulsar_data = {
    # Reference PTA - parameters from this PTA will be inherited by the MetaPulsar
    # for model components that are merged (astrometry, spindown, binary, dispersion)
    "epta_dr2": [
        {
            "par": "../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200.par",
            "tim": "../../data/ipta-dr2/EPTA_v2.2/J0613-0200/J0613-0200_all.tim",
            "timing_package": "tempo2",  # Timing package used
        }
    ],
    "nanograv_9y": [
        {
            "par": "../../data/ipta-dr2/NANOGrav_9y/par/J0613-0200_NANOGrav_9yv1.gls.par",
            "tim": "../../data/ipta-dr2/NANOGrav_9y/tim/J0613-0200_NANOGrav_9yv1.tim",
            "timing_package": "pint",
        }
    ],
    "ppta_dr2": [
        {
            "par": "../../data/ipta-dr2/PPTA_dr1dr2/par/J0613-0200_dr1dr2.par",
            "tim": "../../data/ipta-dr2/PPTA_dr1dr2/tim/J0613-0200_dr1dr2.tim",
            "timing_package": "tempo2",
        }
    ],
}


### MetaPulsar Creation with Consistent Strategy

The `consistent` strategy merges parameters from different PTAs where possible, inheriting values from the reference PTA for merged model components. While MetaPulsar uses both libstempo/tempo2 and PINT under the hood simultaneously, it uses the full timing model parsing capabilities of PINT under the hood to decide how to merge the timing models. Model components and parameters that PINT does not know about are automatically preserved and treated as 'detector-specific'.

### Combination Strategy Options
- `consistent`: Merge compatible parameters, inherit reference PTA values
- `composite`: Keep all parameters separate with PTA suffixes

### Default mergeable Components
- `astrometry`: Position, proper motion, parallax
- `spindown`: Spin frequency and derivatives
- `binary`: Binary orbital parameters (may also use `pulsar_system`)
- `dispersion`: Dispersion measure and derivatives

Any other PINT TimingModel `category` is possible to use in principle, but the above components make astrophysical sense and are included by default.

#### Consistent is the only ''valid'' method
When creating a `consistent` MetaPulsar, the pulsar really only has a single sky position, binary model, and other relevant Astrophysical models. It's a truly unified model. Instead, the `composite` combination method keeps all parameters separate, which is not physical. Each pulsar dataset from each PTA then has separate model parameters, which is not physical. That approach is sometimes called the ''Borg method'' or the ''Frankenstat'' method in the community.

In [ ]:
# Create MetaPulsar using the 'consistent' strategy
# This merges parameters from different PTAs where possible
# Both PINT and libstempo are run simultaneously under the hood
# We run libstempo in 'sandbox' mode, so our kernel does not crash
metapulsar = create_metapulsar(
    file_data=pulsar_data,
    combination_strategy="consistent",  # Merge compatible parameters
    combine_components=[
        "astrometry",
        "spindown",
        "binary",
        "dispersion",
    ],  # Components to merge
    parfile_output_dir='./parfiles',
    add_dm_derivatives=True,  # Ensure DM1, DM2 are present
)

print(f"Created MetaPulsar: {metapulsar.name}")
print(f"Reference PTA: {list(pulsar_data.keys())[0]}")
print("Combination strategy: consistent")
print("Components merged: astrometry, spindown, binary, dispersion")


## Part 2: MetaPulsar is an Enterprise Pulsar

The resulting MetaPulsar is a fully functional Enterprise pulsar with all standard attributes. It combines data from multiple PTAs into a single unified object.

### Key Features
- **Combined dataset**: All observations combined
- **Merged Astrophysics**: Only a single Astrophysical model
- **PTA-Specific Parameters**: Detector-specific parameters retain PTA suffixes
- **Full Enterprise Compatibility**: Works with all Enterprise analysis tools


In [ ]:
# The resulting MetaPulsar is an Enterprise pulsar with all standard attributes
print("MetaPulsar Enterprise attributes:")
print(f"  Name: {metapulsar.name}")
print(f"  Number of pulsars: {len(metapulsar._pulsars)}")
print(f"  PTA names: {list(metapulsar._pulsars.keys())}")
print(f"  Combination strategy: {metapulsar.combination_strategy}")
print(f"  Model components merged: {metapulsar.combine_components}")

# Show some basic Enterprise pulsar attributes
print("\nEnterprise pulsar attributes:")
print(f"  Number of TOAs: {len(metapulsar.toas)}")
print(
    f"  Frequency range: {metapulsar.freqs.min():.2f} - {metapulsar.freqs.max():.2f} MHz"
)
print(f"  Time span: {(metapulsar.toas.max() - metapulsar.toas.min())/86400.0:.1f} days")

print(
    "\nThe MetaPulsar combines data from multiple PTAs into a single Enterprise pulsar."
)
print(
    f"Merged parameters inherit values from the reference PTA ({list(pulsar_data.keys())[0]})."
)
print("PTA-specific parameters retain their original PTA-specific values.")


### Parameter Naming Conventions

MetaPulsar uses a clear naming convention to distinguish between merged and PTA-specific parameters:

- **Merged parameters**: No suffix, inherit reference PTA values
- **PTA-specific parameters**: Retain PTA suffix (e.g., `_nanograv_9y`, `_epta_dr2`)

This allows you to easily identify which parameters are shared across PTAs and which are specific to individual PTAs.


In [ ]:
# Demonstrate parameter naming conventions
print("Parameter naming conventions:")
print("Merged parameters (no suffix):")
fitparams = metapulsar.fitpars
# Get PTA names from our data structure
pta_names = list(pulsar_data.keys())
pta_suffixes = [f"_{pta}" for pta in pta_names]

merged_params = [
    p for p in fitparams if not any(suffix in p for suffix in pta_suffixes)
]
for param in merged_params[:5]:  # Show first 5 merged parameters
    print(f"  {param}")

print("\nPTA-specific parameters (retain PTA suffix):")
pta_specific_params = [
    p for p in fitparams if any(suffix in p for suffix in pta_suffixes)
]
for param in pta_specific_params[:5]:  # Show first 5 PTA-specific parameters
    print(f"  {param}")

print("\nThis naming convention allows you to distinguish between:")
print("  - Merged parameters: Inherit reference PTA values, no suffix")
print("  - PTA-specific parameters: Retain original values, keep PTA suffix")


## Part 3: Automated Multi-Pulsar Processing

For processing multiple pulsars, manually creating data structures becomes cumbersome. MetaPulsar provides utility functions based on regex pattern matching for automation.

### Automated Workflow
1. **Discover layouts**: Automatically detect data release directory structures
2. **Combine layouts**: Merge discovered layouts with predefined patterns
3. **Discover files**: Find all pulsar files using pattern matching
4. **Create MetaPulsars**: Process discovered pulsars automatically (limited to 3 for performance)

### Reference PTA Selection Strategies
- **Auto-selection**: Choose PTA with longest timespan per pulsar
- **Global reference**: Use same PTA as reference for all pulsars
- **Manual**: Reorder PTAs for specific pulsars

**Note**: For demonstration purposes, we limit processing to the first 3 pulsars to keep execution time reasonable.


### Directory Layout Discovery

In [ ]:
# Directory layout is different for each PTA data release
# We provide a regexp discovery engine that helps. It's not perfect,
# but it works for the PTA data releases we have seen so far.

epta_layout = discover_layout('../../data/ipta-dr2/EPTA_v2.2', name='EPTA dr2')
ppta_layout = discover_layout('../../data/ipta-dr2/PPTA_dr1dr2', name='PPTA dr1dr2')
nanograv_layout = discover_layout('../../data/ipta-dr2/NANOGrav_9y', name='NANOGrav 9y')

# Combine layouts with:
combined_layout = combine_layouts(epta_layout, ppta_layout, nanograv_layout)

In [ ]:
# The pre-defined PTA data release regexp patterns are:
from metapulsar import PTA_DATA_RELEASES
PTA_DATA_RELEASES.keys(), PTA_DATA_RELEASES['epta_dr2']

#### File discovery
Those regexp patterns can be fed to the file discovery service. There are some default regexp expressions as well, but it's unlikely that they match the exact structure you have in your data release

In [ ]:
file_data = discover_files(combined_layout)

In [ ]:
# If we want to see a more detailed summary of the PTA data release, we can use the pta_summary function
from metapulsar import pta_summary

pta_summary(file_data)

# Filtering for pulsars

The file_data is just a dictionary with file path information. The pulsars are _not_ yet matched between PTAs. This is not done by name, but by coordinate. We provide some utility functions to see which pulsars are in this file structure

In [ ]:
from metapulsar import get_pulsar_names_from_file_data, filter_file_data_by_pulsars

In [ ]:

# Do coordinate-based pulsar matching between PTAs
pulsar_names = get_pulsar_names_from_file_data(file_data)

len(pulsar_names), pulsar_names[:5]

In [ ]:
# We filter to only include 3 pulsars
# In this step we can choose both the 'B' and the 'J' name! The names are resolved automagically by coordinates. Honeybadger don't care.
pulsar_selection = ['B1855+09', 'J1939+2135', 'J0030+0451']

filtered_data = filter_file_data_by_pulsars(file_data, pulsar_selection)
filtered_data

# Part 2: Create MetaPulsars for all discovered pulsars

We can now create MetaPulsars for all discovered pulsars.
We can either auto-select a reference PTA for each pulsar, or use a specific PTA for all pulsars.

### Reference PTA
The reference PTA is the PTA for which the par-file values are taken as 'reference values'. While these parameters are marginalized over, we do need to gauge them and lock them together. This is done by choosing one of the PTAs, and using those values in the par files. This choice is usually not important for the outcome, especially for millisecond pulsars. However, it's wise to choose the most accurate model for it. Usually, the longest PTA dataset is a good bet.


In [ ]:
from metapulsar import create_all_metapulsars, filter_file_data_by_pulsars

In [ ]:
metapulsars = create_all_metapulsars(filtered_data, reference_pta=None) #='NANOGrav 9y')
# The default options are:
# combination_strategy="consistent",
# combine_components=["astrometry", "spindown", "binary", "dispersion"],
# add_dm_derivatives=True,

# The reference_pta=None option means that the reference PTA will be auto-selected for each pulsar based on observation timespans.

In [ ]:
# Now we have a full list of MetaPulsars
# Pulsars by default are listed by their 'B' name if at least one of the PTAs uses that convention
metapulsars

In [ ]:
metapulsars['B1855+09'].name

In [ ]:
# Checks against MetaPulsar legacy

In [ ]:
from metapulsar.legacy import metapulsar as legacy_module
import numpy as np

# Get all available pulsars from the combined file data
pulsar_names = get_pulsar_names_from_file_data(file_data)
print(f"Found {len(pulsar_names)} pulsars: {pulsar_names[:10]}...")

# Test results tracking
test_results = []
failed_pulsars = []

# Test first 5 pulsars to keep it manageable
test_pulsars = pulsar_names #[:5]

def compare_arrays(arr1, arr2, name, rtol=1e-10, atol=1e-12):
    """Compare two arrays with detailed error reporting."""
    if len(arr1) != len(arr2):
        return False, f"Length mismatch: {len(arr1)} vs {len(arr2)}"
    
    if not np.allclose(arr1, arr2, rtol=rtol, atol=atol):
        max_diff = np.max(np.abs(arr1 - arr2))
        mean_diff = np.mean(np.abs(arr1 - arr2))
        return False, f"Values differ: max_diff={max_diff:.2e}, mean_diff={mean_diff:.2e}"
    
    return True, "Match"

def compare_parameter_names(legacy_fitpars, new_fitpars):
    """Compare parameter names using sets."""
    legacy_set = set(legacy_fitpars)
    new_set = set(new_fitpars)
    
    if legacy_set == new_set:
        return True, "Match"
    else:
        missing_in_new = legacy_set - new_set
        extra_in_new = new_set - legacy_set
        return False, f"Missing in new: {missing_in_new}, Extra in new: {extra_in_new}"

def compare_design_matrices(legacy_dm, new_dm, legacy_fitpars, new_fitpars, legacy_isort, new_isort):
    """Compare design matrices with proper reordering."""
    # Check shapes first
    if legacy_dm.shape != new_dm.shape:
        return False, f"Shape mismatch: {legacy_dm.shape} vs {new_dm.shape}"
    
    # Create mapping from new parameter order to legacy parameter order
    try:
        new_to_legacy_indices = [new_fitpars.index(param) for param in legacy_fitpars]
    except ValueError as e:
        return False, f"Parameter mapping failed: {e}"
    
    # Reorder new design matrix columns to match legacy order
    new_dm_reordered = new_dm[:, new_to_legacy_indices]
    
    # Reorder rows (TOAs) using isort
    legacy_dm_sorted = legacy_dm[legacy_isort, :]
    new_dm_sorted = new_dm_reordered[new_isort, :]
    
    # Compare design matrix values
    return compare_arrays(legacy_dm_sorted, new_dm_sorted, "Design Matrix", rtol=1e-2, atol=1e-5)

def compare_flags(legacy_flags, new_flags):
    """Compare flags with normalization for timing_package case sensitivity."""
    if len(legacy_flags) != len(new_flags):
        return False, f"Length mismatch: {len(legacy_flags)} vs {len(new_flags)}"
    
    # Normalize timing_package field to handle case sensitivity differences
    legacy_flags_normalized = legacy_flags.copy()
    new_flags_normalized = new_flags.copy()
    
    # Convert timing_package to lowercase for comparison
    legacy_flags_normalized["timing_package"] = np.char.lower(legacy_flags["timing_package"])
    new_flags_normalized["timing_package"] = np.char.lower(new_flags["timing_package"])
    
    if not np.array_equal(legacy_flags_normalized, new_flags_normalized):
        # Find differences
        diff_fields = []
        for field in legacy_flags.dtype.names:
            if not np.array_equal(legacy_flags_normalized[field], new_flags_normalized[field]):
                diff_fields.append(field)
        return False, f"Flag differences in fields: {diff_fields}"
    
    return True, "Match"

for pulsar_name in test_pulsars:
    print(f"\n=== Testing pulsar: {pulsar_name} ===")
    
    try:
        # Filter file data for this specific pulsar
        single_pulsar_data = filter_file_data_by_pulsars(file_data, pulsar_name)
        
        # Create new-style MetaPulsar using create_metapulsar
        new_metapulsar = create_metapulsar(
            file_data=single_pulsar_data,
            combination_strategy="consistent",
            reference_pta="NANOGrav_9y",  # Set NANOGrav as reference as requested
            combine_components=["astrometry", "spindown", "binary", "dispersion"],
            add_dm_derivatives=True
        )
        
        # Prepare input files for legacy MetaPulsar creation
        # Convert file_data format to legacy input format
        legacy_input_files = []
        
        for pta_name, files in single_pulsar_data.items():
            if not files:
                continue
                
            # Get the first (and should be only) file for this PTA
            file_info = files[0]
            
            # Determine timing package based on PTA
            timing_package = file_info.get("timing_package", "tempo2")
            if pta_name == "NANOGrav_9y":
                timing_package = "pint"
            elif pta_name in ["EPTA_v2.2", "PPTA_dr1dr2"]:
                timing_package = "tempo2"
            
            legacy_input_files.append({
                "pta": pta_name,
                "parfile": str(file_info["par"]),  # Convert Path to string
                "timfile": str(file_info["tim"]),  # Convert Path to string
                "package": timing_package
            })
        
        # Create legacy MetaPulsar
        legacy_metapulsar = legacy_module.create_metapulsar(legacy_input_files)
        
        # Comprehensive comparison
        comparison_results = {}
        all_passed = True
        
        # 1. Compare residuals
        residuals_match, residuals_msg = compare_arrays(
            new_metapulsar._residuals, 
            legacy_metapulsar._residuals, 
            "Residuals"
        )
        comparison_results["residuals"] = (residuals_match, residuals_msg)
        if not residuals_match:
            all_passed = False
        
        # 2. Compare parameter names
        param_names_match, param_names_msg = compare_parameter_names(
            legacy_metapulsar.fitpars, 
            new_metapulsar.fitpars
        )
        comparison_results["parameter_names"] = (param_names_match, param_names_msg)
        if not param_names_match:
            all_passed = False
        
        # 3. Compare design matrices (only if parameter names match)
        if param_names_match:
            design_matrix_match, design_matrix_msg = compare_design_matrices(
                legacy_metapulsar._designmatrix,
                new_metapulsar._designmatrix,
                legacy_metapulsar.fitpars,
                new_metapulsar.fitpars,
                legacy_metapulsar.isort,
                new_metapulsar.isort
            )
            comparison_results["design_matrix"] = (design_matrix_match, design_matrix_msg)
            if not design_matrix_match:
                all_passed = False
        else:
            comparison_results["design_matrix"] = (False, "Skipped due to parameter name mismatch")
            all_passed = False
        
        # 4. Compare flags
        flags_match, flags_msg = compare_flags(
            legacy_metapulsar._flags, 
            new_metapulsar._flags
        )
        comparison_results["flags"] = (flags_match, flags_msg)
        if not flags_match:
            all_passed = False
        
        # 5. Compare TOA errors
        toaerrs_match, toaerrs_msg = compare_arrays(
            new_metapulsar._toaerrs, 
            legacy_metapulsar._toaerrs, 
            "TOA Errors"
        )
        comparison_results["toaerrs"] = (toaerrs_match, toaerrs_msg)
        if not toaerrs_match:
            all_passed = False
        
        # 6. Compare TOAs (bonus check)
        toas_match, toas_msg = compare_arrays(
            new_metapulsar._toas, 
            legacy_metapulsar._toas, 
            "TOAs"
        )
        comparison_results["toas"] = (toas_match, toas_msg)
        if not toas_match:
            all_passed = False
        
        # Print detailed results
        print("  📊 Comparison Results:")
        for component, (passed, message) in comparison_results.items():
            status = "✅" if passed else "❌"
            print(f"    {status} {component}: {message}")
        
        # Print shapes for reference
        print("  📏 Shapes:")
        print(f"    New - Residuals: {new_metapulsar._residuals.shape}, Design Matrix: {new_metapulsar._designmatrix.shape}, Flags: {len(new_metapulsar._flags)}")
        print(f"    Legacy - Residuals: {legacy_metapulsar._residuals.shape}, Design Matrix: {legacy_metapulsar._designmatrix.shape}, Flags: {len(legacy_metapulsar._flags)}")
        print(f"    New fitpars: {len(new_metapulsar.fitpars)}, Legacy fitpars: {len(legacy_metapulsar.fitpars)}")
        
        if all_passed:
            print(f"  🎉 All comparisons passed for {pulsar_name}")
            test_results.append((pulsar_name, True, "All comparisons passed"))
        else:
            print(f"  ❌ Some comparisons failed for {pulsar_name}")
            failed_components = [comp for comp, (passed, _) in comparison_results.items() if not passed]
            test_results.append((pulsar_name, False, f"Failed: {failed_components}"))
            failed_pulsars.append(pulsar_name)
            
    except Exception as e:
        print(f"  ❌ Error testing {pulsar_name}: {str(e)}")
        test_results.append((pulsar_name, False, f"Error: {str(e)}"))
        failed_pulsars.append(pulsar_name)

# Summary
print("\n=== COMPREHENSIVE TEST SUMMARY ===")
print(f"Total pulsars tested: {len(test_pulsars)}")
print(f"Successful comparisons: {len([r for r in test_results if r[1]])}")
print(f"Failed comparisons: {len([r for r in test_results if not r[1]])}")

if failed_pulsars:
    print(f"Failed pulsars: {failed_pulsars}")
else:
    print("🎉 All tests passed! Legacy and new implementations are fully consistent.")

# Detailed results
print("\n=== DETAILED RESULTS ===")
for pulsar, success, message in test_results:
    status = "✅" if success else "❌"
    print(f"{status} {pulsar}: {message}")

# Just check the failed pulsars

In [ ]:
failed_pulsars = ['J1022+1001', 'J1857+0943', 'J1955+2908', 'J2145-0750', 'J1603-7202']

from metapulsar.legacy import metapulsar as legacy_module
from metapulsar import create_metapulsar

# Configure new-style creation options to match your comparison cell
REFERENCE_PTA = "NANOGrav_9y"
COMBINE_COMPONENTS = ["astrometry", "spindown", "binary", "dispersion"]

def build_legacy_inputs(single_pulsar_data):
    legacy_input_files = []
    for pta_name, files in single_pulsar_data.items():
        if not files:
            continue
        file_info = files[0]  # assume one file per PTA
        timing_package = file_info.get("timing_package", "tempo2")
        if pta_name == "NANOGrav_9y":
            timing_package = "pint"
        elif pta_name in ["EPTA_v2.2", "PPTA_dr1dr2"]:
            timing_package = "tempo2"
        legacy_input_files.append({
            "pta": pta_name,
            "parfile": str(file_info["par"]),
            "timfile": str(file_info["tim"]),
            "package": timing_package,
        })
    return legacy_input_files

def load_metapulsar_pair(pulsar_name, file_data):
    single_pulsar_data = filter_file_data_by_pulsars(file_data, pulsar_name)

    new_mp = create_metapulsar(
        file_data=single_pulsar_data,
        combination_strategy="consistent",
        reference_pta=REFERENCE_PTA,
        combine_components=COMBINE_COMPONENTS,
        add_dm_derivatives=True,
        parfile_output_dir="./parfiles/"
    )

    legacy_inputs = build_legacy_inputs(single_pulsar_data)
    legacy_mp = legacy_module.create_metapulsar(legacy_inputs, par_output_dir="./parfiles/")

    return new_mp, legacy_mp

# Build for all failed pulsars
#metapulsar_pairs = {}  # { pulsar_name: {"new": new_mp, "legacy": legacy_mp} }
#for name in failed_pulsars:
#    new_mp, legacy_mp = load_metapulsar_pair(name, file_data)
#    metapulsar_pairs[name] = {"new": new_mp, "legacy": legacy_mp}

# Example access:
# metapulsar_pairs[failed_pulsars[0]]["new"]
# metapulsar_pairs[failed_pulsars[0]]["legacy"]

In [ ]:
mpsr_new, mpsr_legacy = load_metapulsar_pair(failed_pulsars[0], file_data)

In [ ]:
set(mpsr_new.fitpars) - set(mpsr_legacy.fitpars), set(mpsr_legacy.fitpars) - set(mpsr_new.fitpars)

In [ ]:
[fp for fp in mpsr_new.fitpars if fp.startswith("JUMP")]

In [ ]:
[fp for fp in mpsr_legacy.fitpars if fp.startswith("JUMP")]

In [ ]:
mpsr_new.fitpars.index('JUMP1_PPTA dr1dr2')

In [ ]:
np.sum(np.abs(mpsr_new._designmatrix[:,30]))

In [ ]:
mpsr_new._fitparameters

# Position checks

In [ ]:
# The pulsars with failed position tests
failed_pulsars = ['J1022+1001', 'J1857+0943', 'J1955+2908', 'J2145-0750', 'J1603-7202']

In [ ]:
import numpy as np
from io import StringIO
from astropy.coordinates import SkyCoord, FK4, ICRS, BarycentricTrueEcliptic, Angle
from astropy.time import Time
import astropy.units as u
from pint.models.model_builder import parse_parfile

def _first_val(v):
    return v[0] if isinstance(v, list) and v else (v if isinstance(v, str) else "")

def _load_par_dict(par_path):
    with open(par_path, "r") as f:
        pd = parse_parfile(StringIO(f.read()))
    # flatten to first token of first value (drop fit flags/uncertainty)
    d = {}
    for k, v in pd.items():
        s = _first_val(v)
        d[k] = s.split()[0] if isinstance(s, str) else s
    return d

def _correct_icrs_from_par(pd):
    # Prefer RAJ/DECJ (ICRS/J2000)
    if pd.get("RAJ") and pd.get("DECJ"):
        return SkyCoord(ra=Angle(pd["RAJ"], unit=u.hourangle),
                        dec=Angle(pd["DECJ"], unit=u.deg),
                        frame=ICRS())
    # Ecliptic fallback
    lam = pd.get("LAMBDA") or pd.get("ELONG")
    bet = pd.get("BETA") or pd.get("ELAT")
    if lam and bet:
        c_ecl = SkyCoord(lon=Angle(lam, unit=u.deg),
                         lat=Angle(bet, unit=u.deg),
                         distance=1*u.pc,
                         frame=BarycentricTrueEcliptic(equinox=Time("J2000")))
        return c_ecl.transform_to(ICRS())
    # FK4 B1950 fallback
    if pd.get("RA") and pd.get("DEC"):
        c_fk4 = SkyCoord(ra=Angle(pd["RA"], unit=u.hourangle),
                         dec=Angle(pd["DEC"], unit=u.deg),
                         frame=FK4(equinox=Time("B1950")))
        return c_fk4.transform_to(ICRS())
    raise ValueError("No usable coordinates found in parfile")

def _legacy_icrs_from_par(pd, name):
    # Emulate legacy: if name starts with 'B', treat coords as FK4/B1950 regardless
    if name.startswith("B"):
        if pd.get("RAJ") and pd.get("DECJ"):
            # WRONG on purpose: misinterpret RAJ/DECJ as FK4 B1950 then convert
            c_b = SkyCoord(ra=Angle(pd["RAJ"], unit=u.hourangle),
                           dec=Angle(pd["DECJ"], unit=u.deg),
                           frame=FK4(equinox=Time("B1950")))
            c_fk4j = c_b.transform_to(FK4(equinox=Time("J2000")))
            return c_fk4j.transform_to(ICRS())
        if pd.get("RA") and pd.get("DEC"):
            c_b = SkyCoord(ra=Angle(pd["RA"], unit=u.hourangle),
                           dec=Angle(pd["DEC"], unit=u.deg),
                           frame=FK4(equinox=Time("B1950")))
            c_fk4j = c_b.transform_to(FK4(equinox=Time("J2000")))
            return c_fk4j.transform_to(ICRS())
    # Otherwise, use correct handling
    return _correct_icrs_from_par(pd)

def diagnose_legacy_position_issue(parfiles: dict, atol_arcsec=0.021):
    """
    parfiles: dict like {"EPTA": "/path/to.par", "PPTA": "/path/to.par"}
    atol_arcsec: legacy absolute tolerance ~ 1e-7 rad ≈ 0.021"
    """
    rows = []
    first_correct = None
    first_legacy = None

    for label, path in parfiles.items():
        pd = _load_par_dict(path)
        name = pd.get("PSR") or pd.get("PSRJ") or pd.get("PSRB") or "UNKNOWN"
        has_rajdecj = bool(pd.get("RAJ") and pd.get("DECJ"))
        has_radec = bool(pd.get("RA") and pd.get("DEC"))
        has_ecl = bool((pd.get("LAMBDA") or pd.get("ELONG")) and (pd.get("BETA") or pd.get("ELAT")))

        c_corr = _correct_icrs_from_par(pd)
        c_leg = _legacy_icrs_from_par(pd, name)

        if first_correct is None:
            first_correct = c_corr
            first_legacy = c_leg

        d_ra_corr = (c_corr.ra - first_correct.ra).to(u.arcsec).value
        d_dec_corr = (c_corr.dec - first_correct.dec).to(u.arcsec).value
        d_ra_leg = (c_leg.ra - first_legacy.ra).to(u.arcsec).value
        d_dec_leg = (c_leg.dec - first_legacy.dec).to(u.arcsec).value

        ok_corr = (abs(d_ra_corr) <= atol_arcsec) and (abs(d_dec_corr) <= atol_arcsec)
        ok_leg = (abs(d_ra_leg) <= atol_arcsec) and (abs(d_dec_leg) <= atol_arcsec)

        rows.append({
            "label": label, "name": name,
            "has_RAJDECJ": has_rajdecj, "has_RADEC": has_radec, "has_ecl": has_ecl,
            "dRA_correct_arcsec": d_ra_corr, "dDec_correct_arcsec": d_dec_corr, "within_tol_correct": ok_corr,
            "dRA_legacy_arcsec": d_ra_leg, "dDec_legacy_arcsec": d_dec_leg, "within_tol_legacy": ok_leg,
        })

    print(f"Tolerances: atol={atol_arcsec:.3f}\"")
    for r in rows:
        print(f"[{r['label']}] {r['name']}  RAJ/DECJ={r['has_RAJDECJ']} RA/DEC={r['has_RADEC']} ECL={r['has_ecl']}")
        print("  correct ΔRA={:.4f}\" ΔDec={:.4f}\"  tol_ok={}".format(r["dRA_correct_arcsec"], r["dDec_correct_arcsec"], r["within_tol_correct"]))
        print("  legacy  ΔRA={:.4f}\" ΔDec={:.4f}\"  tol_ok={}".format(r["dRA_legacy_arcsec"], r["dDec_legacy_arcsec"], r["within_tol_legacy"]))

    # Heuristic: flag RAJ/DECJ present with B-name (legacy misinterpretation)
    if any(r["name"].startswith("B") and r["has_RAJDECJ"] for r in rows):
        print("\nLikely bug: B-name with RAJ/DECJ present; legacy path misinterprets RAJ/DECJ as B1950.")
    return rows

# Usage:
# parfiles = {
#     "EPTA": "/abs/path/to/EPTA.par",
#     "PPTA": "/abs/path/to/PPTA.par",
# }
# diagnose_legacy_position_issue(parfiles)

In [ ]:
failed_pulsars = ['J1857+0943', 'J1955+2908']
single_pulsar_data = filter_file_data_by_pulsars(file_data, failed_pulsars[0])
#single_pulsar_data
par_files = {}
for pta_name, files in single_pulsar_data.items():
    if not files:
        continue
    file_info = files[0]
    par_files[pta_name] = file_info["par"]

diagnose_legacy_position_issue(par_files)